# Vinyl Taste Profile & Discoveries

An analysis of my record collection — what I listen to, where it clusters, and what the algorithm has surfaced that I don't own yet.

In [ ]:
import json, os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'vscode'  # VS Code built-in Plotly renderer
from pathlib import Path
from collections import defaultdict
from IPython.display import HTML, display

DATA_ROOT       = Path(os.environ.get('DISCOGS_DATA_ROOT', Path.home() / 'DiscogsData'))
PROFILE_PATH    = DATA_ROOT / 'profile/releases.json'
EMBEDDINGS_PATH = DATA_ROOT / 'embeddings/my_collection_embeddings.npz'
TOP_DIR         = DATA_ROOT / 'exports/top10'

profile_data = json.loads(PROFILE_PATH.read_text())
releases     = profile_data['releases']
username     = profile_data['username']

collection = [r for r in releases if r['source'] == 'collection']
wantlist   = [r for r in releases if r['source'] == 'wantlist']

print(f'User: {username}')
print(f'Collection: {len(collection)} releases  |  Wantlist: {len(wantlist)} releases')

---
## 1. Collection DNA

What styles, labels, and eras define the collection. Wantlist items are weighted 1.5× — they signal intent more strongly than existing purchases.

In [ ]:
COLLECTION_WEIGHT = 1.0
WANTLIST_WEIGHT   = 1.5

style_raw  = defaultdict(float)
year_raw   = defaultdict(float)
label_raw  = defaultdict(float)

for r in releases:
    w = COLLECTION_WEIGHT if r['source'] == 'collection' else WANTLIST_WEIGHT
    for s in (r.get('styles') or []):
        style_raw[s] += w
    if r.get('year') and 1970 <= int(r['year']) <= 2026:
        year_raw[int(r['year'])] += w
    for l in (r.get('label_names') or []):
        label_raw[l] += w

# --- Styles ---
style_df = pd.DataFrame(
    sorted(style_raw.items(), key=lambda x: x[1], reverse=True)[:25],
    columns=['Style', 'Weight']
)
fig = px.bar(
    style_df, x='Weight', y='Style', orientation='h',
    title='Top Styles in My Collection & Wantlist',
    color='Weight', color_continuous_scale='Teal',
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False,
    plot_bgcolor='#0e1117', paper_bgcolor='#0e1117', font_color='#e0e0e0',
    title_font_size=18, height=600, margin=dict(l=140),
)
fig.show()

In [ ]:
# --- Year distribution ---
year_df = pd.DataFrame(
    sorted(year_raw.items()), columns=['Year', 'Weight']
)
fig2 = px.bar(
    year_df, x='Year', y='Weight',
    title='Release Year Distribution',
    color='Weight', color_continuous_scale='Teal',
)
fig2.update_layout(
    coloraxis_showscale=False,
    plot_bgcolor='#0e1117', paper_bgcolor='#0e1117', font_color='#e0e0e0',
    title_font_size=18, height=380,
)
fig2.show()

In [ ]:
# --- Top labels ---
label_df = pd.DataFrame(
    sorted(label_raw.items(), key=lambda x: x[1], reverse=True)[:20],
    columns=['Label', 'Weight']
)
fig3 = px.bar(
    label_df, x='Weight', y='Label', orientation='h',
    title='Top Labels in My Collection & Wantlist',
    color='Weight', color_continuous_scale='Teal',
)
fig3.update_layout(
    yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False,
    plot_bgcolor='#0e1117', paper_bgcolor='#0e1117', font_color='#e0e0e0',
    title_font_size=18, height=560, margin=dict(l=200),
)
fig3.show()

---
## 2. The Mean Record

Every record in the collection has an audio embedding — a 1280-dimensional fingerprint computed by Essentia's EffNet model trained on the Discogs catalogue. The centroid is the average of all those fingerprints. The track below is the one whose audio is closest to that centroid: the most *typical* sounding track in the collection.

In [ ]:
emb_data   = np.load(EMBEDDINGS_PATH)
embeddings = emb_data['embeddings'].astype(np.float32)  # (N, 1280)
centroid   = emb_data['centroid'].astype(np.float32).flatten()
filenames  = emb_data['filenames']

# Cosine similarity of every track to the centroid
centroid_norm = centroid / (np.linalg.norm(centroid) + 1e-8)
norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8
sims  = (embeddings / norms) @ centroid_norm

closest_idx  = int(np.argmax(sims))
closest_sim  = float(sims[closest_idx])
closest_file = filenames[closest_idx]

# Best-effort parse of "Artist - Release - Track" filename pattern
stem  = Path(closest_file).stem
parts = stem.split(' - ', 2)
mean_artist  = parts[0].strip() if len(parts) >= 2 else ''
mean_release = parts[1].strip() if len(parts) >= 2 else stem
mean_track   = parts[2].strip() if len(parts) > 2 else ''

print(f'Tracks embedded:  {len(embeddings)}')
print(f'Cosine sim:       {closest_sim:.4f}')
print(f'Artist:           {mean_artist or "(unknown)"}')
print(f'Release:          {mean_release}')
print(f'Track:            {mean_track or "(full file)"}')
print(f'Raw filename:     {closest_file}')

In [ ]:
# Distribution of all tracks relative to centroid
sim_df = pd.DataFrame({'Cosine similarity': sims})

fig4 = px.histogram(
    sim_df, x='Cosine similarity', nbins=40,
    title='Audio Similarity to Collection Centroid — All Tracks',
    color_discrete_sequence=['#2dd4bf'],
)
fig4.add_vline(
    x=closest_sim, line_dash='dash', line_color='#f97316',
    annotation_text='← mean record', annotation_position='top right',
)
fig4.update_layout(
    plot_bgcolor='#0e1117', paper_bgcolor='#0e1117',
    font_color='#e0e0e0', title_font_size=18, height=380,
)
fig4.show()

---
## 3. Discoveries

Records surfaced by the recommendation pipeline that aren't in the collection. Scored by a blend of label/artist graph affinity and audio similarity to the collection centroid.

**Teal** = affinity pick (connected to known labels/artists) · **Purple** = discovery (new territory)

### Favourites so far

In [ ]:
# Add any records you love here — paste the Discogs & YouTube URLs
FAVOURITES = [
    {
        'title':       'Water',
        'artists':     'Rithma',
        'labels':      'Open Bar Music',
        'styles':      'House, Deep House',
        'country':     'Canada',
        'year':        '2004',
        'note':        'Hypnotic deep groove — standout pick from the first runs.',
        'discogs_url': 'https://www.discogs.com/release/367422',
        'youtube_url': 'https://www.youtube.com/watch?v=Jp1GvlgOxdQ',
        'image_url':   '',
    },
    {
        'title':       'Who Am I?',
        'artists':     'Rossi.',
        'labels':      '',
        'styles':      'Minimal, Dub Techno, Tech House, House',
        'country':     '',
        'year':        '',
        'note':        'Sits right at the intersection of Dub Techno and Minimal.',
        'discogs_url': 'https://www.discogs.com/release/22335796',
        'youtube_url': '',
        'image_url':   '',
    },
    {
        'title':       'Past Lessons / Future Theories',
        'artists':     'Slam',
        'labels':      'Soma Quality Recordings',
        'styles':      'Techno, Tech House',
        'country':     'UK',
        'year':        '',
        'note':        'Hard-edged Techno/Tech House from a legendary Glasgow duo.',
        'discogs_url': 'https://www.discogs.com/release/72446',
        'youtube_url': '',
        'image_url':   '',
    },
]


def _card(r: dict, badge_color: str = '#2dd4bf') -> str:
    if r.get('image_url'):
        img = f'<img src="{r["image_url"]}" style="width:110px;height:110px;object-fit:cover;border-radius:6px;flex-shrink:0">'
    else:
        img = '<div style="width:110px;height:110px;background:#1e293b;border-radius:6px;flex-shrink:0;display:flex;align-items:center;justify-content:center;color:#475569;font-size:32px">♫</div>'

    btns = ''
    if r.get('youtube_url'):
        btns += f'<a href="{r["youtube_url"]}" target="_blank" style="background:#dc2626;color:#fff;padding:5px 12px;border-radius:4px;text-decoration:none;font-size:12px;margin-right:8px">▶ YouTube</a>'
    if r.get('discogs_url'):
        btns += f'<a href="{r["discogs_url"]}" target="_blank" style="background:#1a56db;color:#fff;padding:5px 12px;border-radius:4px;text-decoration:none;font-size:12px">Discogs</a>'

    meta = ' · '.join(filter(None, [r.get('labels',''), r.get('country',''), str(r.get('year',''))[:4]]))
    note = f'<p style="color:#94a3b8;font-style:italic;margin:4px 0 0;font-size:13px">{r["note"]}</p>' if r.get('note') else ''

    return f'''
    <div style="display:flex;gap:16px;background:#1e293b;border-radius:10px;padding:14px;margin-bottom:10px;align-items:flex-start">
      {img}
      <div style="flex:1;min-width:0">
        <div style="font-size:16px;font-weight:700;color:#f1f5f9;margin-bottom:2px">{r["title"]}</div>
        <div style="font-size:14px;color:#94a3b8;margin-bottom:6px">{r["artists"]}</div>
        <span style="background:#0f172a;color:{badge_color};padding:2px 8px;border-radius:99px;font-size:11px;border:1px solid {badge_color}44">{r["styles"]}</span>
        <div style="font-size:12px;color:#64748b;margin-top:5px;margin-bottom:8px">{meta}</div>
        {note}
        <div style="margin-top:10px">{btns}</div>
      </div>
    </div>'''


html = '<div style="font-family: system-ui, sans-serif; max-width: 760px">'
for r in FAVOURITES:
    html += _card(r, badge_color='#f97316')
html += '</div>'
display(HTML(html))

### Latest algorithm run — full results

In [ ]:
top_files = sorted(TOP_DIR.glob('top10_v*.csv'))
if not top_files:
    print('No exports found — run: python3 main.py score --force && python3 main.py audio_rank --verbose')
else:
    latest = top_files[-1]
    df = pd.read_csv(latest)

    html = f'<div style="font-family: system-ui, sans-serif; max-width: 760px">'
    html += f'<p style="color:#64748b;margin-bottom:16px">Source: <code>{latest.name}</code> · {len(df)} records · <span style="color:#2dd4bf">■ affinity</span> &nbsp; <span style="color:#a78bfa">■ discovery</span></p>'

    for _, row in df.iterrows():
        bucket = str(row.get('bucket', 'affinity'))
        color  = '#2dd4bf' if bucket == 'affinity' else '#a78bfa'

        def _parse(val):
            try: return ', '.join(json.loads(str(val)))
            except: return str(val)

        audio_sim = float(row.get('audio_sim', 0))
        rec = {
            'title':       str(row.get('title', '')),
            'artists':     _parse(row.get('artists', '')),
            'labels':      _parse(row.get('labels', '')),
            'styles':      f"{_parse(row.get('styles',''))}  ·  {bucket}  ·  sim {audio_sim:.3f}",
            'country':     str(row.get('country', '')),
            'year':        str(row.get('released', ''))[:4],
            'note':        '',
            'discogs_url': str(row.get('discogs_url', '')),
            'youtube_url': str(row.get('youtube_url', '')),
            'image_url':   str(row.get('image_url', '')),
        }
        html += _card(rec, badge_color=color)

    html += '</div>'
    display(HTML(html))